# Group Project - Safe routes home at night 

**DS105A Group Project – Data for Data Science (Autumn Term 2025/2026)**

<div style="font-family: system-ui; padding: 20px 30px 20px 20px; background-color: #FFFFFF; border-left: 8px solid #ED9255; border-radius: 8px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);max-width:600px;color:#212121;">

**Group Notebook**
- 👤 Team Name: CtrlAltElite
- 🎯 Purpose: This notebook contains code outlining our data collection approach. 
</div>

In [ ]:
import requests
import json
import os
import pandas as pd
from pathlib import Path
import warnings
import math
from collections import defaultdict
import re

## Data Collection Methodology

1. **Using Google Places API, find nightclubs for a certain area**

2. **Collect up to twenty stations within a radius of 1 kilometre from the nightclub:**
    - allows for up to a twelve minute walk at a typical walking pace of 3mph.

3. **Filter stations according to direction**
    - Preserving the labels from step 2, divide the area surrounding a club into eight 45 degree sections. For each section, take _the two closest stations_ to the nightclub **but always include** train stations (typically larger transport hubs).

4. **Generate Routes:**
    - Use Google Routes API to calculate routes for stations (computeroutematrix allows for batch routing, saving costs).
    Label routes accordingly:
        - 'very short' ( under 5 mins).
        - 'short' (between 5-8 mins).
        - 'long' (between 8-15 mins).

5. **Prepare Routes for Police API:**
    - Reduce the number of route vertices in the google maps polyline;
    - Convert the polyline into a polygon by extending the route polyline on either side, making a 'corridor' to use as a custom area for the police API.

## 1. Gather nightclub data for a popular area in London

This [TimeOut list ](https://www.timeout.com/london/clubs/the-best-clubs-in-london) features 5 clubs in Dalston, the most for any area on the list. This test will use Dalston to generate clubs.

In [2]:
# Constants
PLACES_URL = 'https://places.googleapis.com/v1/places:searchText'
API_KEY = os.getenv('API_KEY')
HEADERS = {
    'Content-Type': 'application/json',
    'X-Goog-API-Key': API_KEY,
    'X-Goog-FieldMask': (
        'places.displayName,'
        'places.postalAddress.postalCode,'
        'places.primaryType,'
        'places.location'
    )
}

AREA_NAMES = ['Soho']
AREA_POSTCODES = {
    'Dalston': ('E8', 'N16'),
    'Soho': ('W1F')
}


In [3]:
def validate_clubs(area_name, version):
    """
    Queries Google Places API for nightclubs within a given area, 
    validates them by checking that their primaryType is either a bar 
    or a nightclub and that they postcode is correct, caches results, 
    and returns the filtered list.
    """

    # This section tries to call the API, bringing about one of three possible results:
    ## 1. Successful call, cache data is refreshed.
    ## 2. Failed call, previously-cached data is loaded and used safely.
    ## 3. Failed call, no cached data available. Fails loudly

    cache_dir = Path(f'data/raw/{version}/{area_name.lower()}')
    cache_path = cache_dir / (f'{area_name.lower()}_nightclubs.json')

    try: 

        query = {
            'textQuery': f'nightclubs in {area_name}, London, England'
        }

        response = requests.post(url=PLACES_URL, headers=HEADERS, data=json.dumps(query))

        response.raise_for_status()
        clubs_data = response.json()
    
    except Exception as e:
        if cache_path.exists():
            warnings.warn(
                f'Places API failed for {area_name}, using cached data instead'
            )
            with open(cache_path) as f:
                return json.load(f)

        else:
            raise RuntimeError(
                f'API failed and no cache available for {area_name}'
            ) from e

    # This section validates clubs returned against their primaryType and postcode

    valid_clubs = []
    postcode_prefixes = AREA_POSTCODES.get(area_name)
    if not postcode_prefixes:
        raise ValueError(f'No postcode prefixes defined for {area_name}')

    for club in clubs_data.get('places', []):
        postcode = (
            club
            .get('postalAddress', {})
            .get('postalCode', '')
        )
        primary_type = club.get("primaryType", "")

        if (
            postcode.startswith(postcode_prefixes)
            and primary_type in {"bar", "nightclub"}
        ):
            valid_clubs.append(club)

    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_path, 'w') as f:
            json.dump(valid_clubs, f, indent=2)

    return valid_clubs

In [26]:
VERSION = 'v1'
dalston_clubs = validate_clubs('Dalston', VERSION)

In [5]:
VERSION = 'v1'
soho_clubs = validate_clubs('Soho', VERSION)

## 2. Collect up to twenty stations within a radius of 1 kilometre from the nightclub

In [6]:
# Test constants

API_KEY = os.getenv('API_KEY')
STATIONS_URL = 'https://places.googleapis.com/v1/places:searchNearby'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'places.types,places.formattedAddress,places.location,places.displayName,places.primaryTypeDisplayName',
    }
ROUNDING = 4 # round location data to avoid unneccesary API calls for similar locations - this allows for roughly a 10m distance 

In [7]:
# define a function to normalize club names for a coherent file system 
def slugify(name: str) -> str:
    name = name.lower()
    name = re.sub(r"[^\w\s-]", "", name)
    name = re.sub(r"\s+", "_", name)
    return name.strip("_")

In [10]:
def get_stations_data(area_name, version):

    clubs_path = Path(f'data/raw/{version}/{area_name.lower()}/{area_name.lower()}_nightclubs.json')

    if not clubs_path.exists():
        raise FileNotFoundError(
            f'No nightclub cache found for {area_name} at {clubs_path}'
        )
    
    with open(clubs_path, 'r') as f:
        clubs = json.load(f)

    output_dir = Path(f"data/raw/{version}/{area_name.lower()}")
    output_dir.mkdir(parents=True, exist_ok=True)

    for club in clubs:
        club_name = club['displayName']['text']
        club_slug = slugify(club_name)

        lat = round(club['location']['latitude'], ROUNDING)
        lon = round(club['location']['longitude'], ROUNDING)

        cache_path = output_dir / f'{club_slug}_stations.json'

        if cache_path.exists():
            continue

        data = {
                'includedTypes' : ['bus_station', 'subway_station', 'train_station'],
                'maxResultCount' : 20,
                'rankPreference': 'DISTANCE',
                'locationRestriction' : {
                    'circle': {
                        'center' : {
                            'latitude' : lat,
                            'longitude' : lon
                        },
                        'radius' : 1000
                    }
                }
            }
            

        try: 
            response = requests.post(url=STATIONS_URL, headers=HEADERS, json=data, timeout=10)
            response.raise_for_status()
            stations_data = response.json()

        except Exception as e:
            warnings.warn(
                f'Failed to fetch stations for {club_name}: {e}'
            )
            continue

        output = {
            'area': area_name,
            'club': club_name,
            'club_location': {
                'latitude': lat,
                'longitude': lon,
            },
            'places': stations_data.get('places', []),
        }

        with open(cache_path, 'w') as f:
            json.dump(output, f, indent=2)

In [21]:
VERSION = 'v1'
dalston_stations = get_stations_data('Dalston', VERSION)

In [11]:
soho_stations = get_stations_data('Soho', VERSION)

## 3. Filter stations according to direction

In [12]:
# define a function that returns a bearing fro point 1 to point 2 in degrees (0-360) where 0 = North and 90 = East

def bearing_degrees(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_lon = math.radians(lon2 - lon1)

    x = math.sin(delta_lon) * math.cos(phi2)
    y = (
        math.cos(phi1) * math.sin(phi2)
        - math.sin(phi1) * math.cos(phi2) * math.cos(delta_lon)
    )

    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

In [13]:
# define a function that assigns each station to one of 8 sectors

def sector_index(bearing):
    return int(bearing // 45)

In [14]:
# define haversine to calculate the shortest distance between two points on a sphere

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius in meters

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [15]:
def directional_filtering(stations, club_lat, club_lon, max_per_sector=2):
    sectors = defaultdict(list)
    train_stations = []

    for station in stations['places']:
        lat = station['location']['latitude']
        lon = station['location']['longitude']

        # 1. compute straight-line distance in metres from club to station

        station['distance_m'] = haversine(club_lat, club_lon, lat, lon)

        # 2. assign station to a bearing sector relative to the club

        bearing = bearing_degrees(club_lat, club_lon, lat, lon)
        sector = sector_index(bearing)

        station['bearing'] = bearing
        station['sector'] = sector
        
        # 3. split by station type

        if station['types'][0] in ('train_station', 'subway_station'):
            train_stations.append(station)
        else:
            sectors[sector].append(station)

    selected = []

    # select closest bus stations per sector

    for sector_stations in sectors.values():
        sector_stations.sort(key=lambda s: s['distance_m'])
        selected.extend(sector_stations[:max_per_sector])

    # always include train/subway stations

    selected_ids = {
        (
            s['location']['latitude'], s['location']['longitude']
            ) 
            for s in selected
        }

    for station in train_stations:
        key = (station['location']['latitude'], station['location']['longitude'])
        if key not in selected_ids:
            selected.append(station)

    return selected

In [16]:
def filter_stations_for_area(area_name, version, max_per_sector=2):
    
    area_dir = Path(f'data/raw/{version}/{area_name.lower()}')

    if not area_dir.exists():
        raise FileNotFoundError(f'No station directory found at {area_dir}')

    output_dir = area_dir

    for station_file in area_dir.glob('*_stations.json'):
        with open(station_file) as f:
            data = json.load(f)

        club_lat = data['club_location']['latitude']
        club_lon = data['club_location']['longitude']

        try:
            filtered = directional_filtering(
                stations=data,
                club_lat=club_lat,
                club_lon=club_lon,
                max_per_sector=max_per_sector,
            )
        except Exception as e:
            warnings.warn(
                f'Filtering failed for {station_file.name}: {e}'
            )

            continue 

        output = {
            'area': data['area'],
            'club': data['club'],
            'club_location': data['club_location'],
            'selected_stations': filtered,
        }

        out_path = output_dir / station_file.name.replace(
            '_stations.json', '_stations_filtered.json'
        )

        with open(out_path, 'w') as f:
            json.dump(output, f, indent=2)


In [34]:
VERSION = 'v1'

filter_stations_for_area(
    area_name = 'Dalston',
    version = VERSION,
    max_per_sector=2
)

In [17]:
filter_stations_for_area(
    area_name = 'Soho',
    version = VERSION,
    max_per_sector=2
)

## 4. Generate Routes for Stations

In [18]:
# Test constants

API_KEY = os.getenv('API_KEY')
ROUTES_URL = 'https://routes.googleapis.com/directions/v2:computeRoutes'
HEADERS = {
        'Content-Type': 'application/json',
        'X-Goog-API-Key': API_KEY,
        'X-Goog-FieldMask': 'routes.duration,routes.distanceMeters,routes.polyline',
    }

In [19]:
def collect_routes_for_club(area_name, club_slug, version):

    base_dir = Path(f'data/raw/{version}/{area_name.lower()}')
    stations_path = base_dir/ f'{club_slug}_stations_filtered.json'
    routes_path = base_dir / f'{club_slug}_routes.json'

    if not stations_path.exists():
        raise FileNotFoundError(f'Missing stations file {stations_path}')

    with open(stations_path, 'r') as f:
        stations_data = json.load(f)

    club_lat = stations_data['club_location']['latitude']
    club_lon = stations_data['club_location']['longitude']

    routes_output = {
    'area': area_name,
    'club': stations_data['club'],
    'club_location': stations_data['club_location'],
    'routes': []
    }

    for station in stations_data.get('selected_stations', []):

        station_lat = station['location']['latitude']
        station_lon = station['location']['longitude']

        data = {
            'origin': {
                'location': {
                    'latLng': {
                        'latitude': club_lat,
                        'longitude': club_lon
                    }
                }
            },
            'destination': {
                'location': {
                    'latLng': {
                        'latitude': station_lat,
                        'longitude': station_lon
                    }
                }
            },
            'travelMode': 'WALK',
            'polylineEncoding': 'GEO_JSON_LINESTRING',
            'polylineQuality': 'OVERVIEW'
        }

        try:
            response = requests.post(
                ROUTES_URL,
                headers=HEADERS,
                json=data,
                timeout=10
            )
            response.raise_for_status()
            route = response.json()['routes'][0]

            routes_output['routes'].append({
                'station_name': station['displayName']['text'],
                'station_location': station['location'],
                'distanceMeters': route['distanceMeters'],
                'duration': route['duration'],
                'polyline': route['polyline']['geoJsonLinestring']['coordinates']
            })

        except Exception as e:
            warnings.warn(
                f'Route failed for {stations_data['club']}'
                f'{station['displayName']['text']}: {e}'
            )

    # Cache routes separately
    with open(routes_path, 'w') as f:
        json.dump(routes_output, f, indent=2)

    return routes_output

In [6]:
VERSION = 'v1'
AREA = 'Dalston'
base_dir = Path(f'data/raw/{VERSION}/{AREA.lower()}')
club_slugs = []

for file_path in base_dir.iterdir():
    if file_path.is_file() and file_path.name.endswith('_filtered.json'):
        club_slug = file_path.name.replace('_stations_filtered.json', '')
        club_slugs.append(club_slug)

for club in club_slugs:
    collect_routes_for_club(AREA, club, VERSION)

In [20]:
VERSION = 'v1'
AREA = 'Soho'
base_dir = Path(f'data/raw/{VERSION}/{AREA.lower()}')
club_slugs = []

for file_path in base_dir.iterdir():
    if file_path.is_file() and file_path.name.endswith('_filtered.json'):
        club_slug = file_path.name.replace('_stations_filtered.json', '')
        club_slugs.append(club_slug)

for club in club_slugs:
    collect_routes_for_club(AREA, club, VERSION)

## 5. Prepare Routes for Police API

In [21]:
def build_police_api_param(
    route,
    club_name,
    area_name,
    version,
    step=3,
    corridor_width_m=20,
):

    """
    Builds a Police API polygon param from a Google Routes API response.
    """
    # First, reduce the number of polyline coordinates for good API citizenship when querying the Police API.

    polyline = route['polyline']
    sampled = polyline[::step]

    if sampled[-1] != polyline[-1]:
        sampled.append(polyline[-1])

    # Next, build a corridor polygon from the polyline coords

    left_side = []
    right_side = []

    for i in range(len(sampled) - 1):
        # coordinates switched because Google Routes returns coords as [longitude, latitude]
        lon1, lat1 = sampled[i]
        lon2, lat2 = sampled[i + 1]

        metres_per_deg_lat = 111_320
        metres_per_deg_lon = 111_320 * math.cos(math.radians(lat1))

        dx = lon2 - lon1
        dy = lat2 - lat1

        length = math.hypot(
            dx * metres_per_deg_lon,
            dy * metres_per_deg_lat,
        )

        # extend coordinates laterally to extend the route into a corridor

        ux = -dy * metres_per_deg_lat / length
        uy =  dx * metres_per_deg_lon / length

        offset_lon = (ux * corridor_width_m) / metres_per_deg_lon
        offset_lat = (uy * corridor_width_m) / metres_per_deg_lat

        left_side.append([lat1 + offset_lat, lon1 + offset_lon])
        right_side.append([lat1 - offset_lat, lon1 - offset_lon])

    # close polygon

    lon_end, lat_end = sampled[-1]
    left_side.append([lat_end + offset_lat, lon_end + offset_lon])
    right_side.append([lat_end - offset_lat, lon_end - offset_lon])

    polygon = left_side + right_side[::-1]
    polygon.append(polygon[0])

    # Finally, edit polygon to match Police API format

    param = ":".join(f"{lat},{lon}" for lat, lon in polygon)

    return param

In [22]:
# define a function to create a metadata json for each police API param txt file

def build_route_metadata(
    *,
    area_name: str,
    club_name: str,
    club_location: dict,
    route: dict,
    route_index: int, 
    police_param_filename: str,
):

    # convert data such as 55s into int seconds 

    duration_raw = route.get("duration", "")
    duration_seconds = (
        int(duration_raw.rstrip("s"))
        if isinstance(duration_raw, str) and duration_raw.endswith("s")
        else None
    )

    metadata = {
        "area": area_name,
        "club": {
            "name": club_name,
            "location": {
                "latitude": club_location["latitude"],
                "longitude": club_location["longitude"],
            },
        },
        "destination": {
            "name": route["station_name"],
            "location": {
                "latitude": route["station_location"]["latitude"],
                "longitude": route["station_location"]["longitude"],
            },
        },
        "route": {
            "route_index": route_index,
            "distance_meters": route.get("distanceMeters"),
            "duration_seconds": duration_seconds,
        },
        "police_api": {
            "param_file": police_param_filename,
        },
    }

    return metadata

In [23]:
def batch_build_police_api_params(
    area_name,
    version,
    corridor_width_m,
):
    """
    Batch-builds Police API polygon params for all clubs in an area directory.
    Saves param as a text file, one file per route
    """

    area_dir = Path(f'data/raw/{version}/{area_name.lower()}')

    if not area_dir.exists():
        raise FileNotFoundError(f"Area directory not found: {area_dir}")

    routes_files = list(area_dir.glob("*_routes.json"))

    if not routes_files:
        raise RuntimeError(f"No route files found in {area_dir}")

    police_params_dir = area_dir / 'police_params'
    metadata_dir = area_dir / 'metadata'

    police_params_dir.mkdir(exist_ok=True)
    metadata_dir.mkdir(exist_ok=True)

    results = {}

    for routes_path in routes_files:
        club_slug = routes_path.stem.replace("_routes", "")
        club_name = club_slug.replace("_", " ").title()

        with open(routes_path) as f:
            data = json.load(f)

        if 'routes' not in data or not data['routes']:
            warnings.warn(f'No routes found in {routes_path.name}')
            continue

        club_location = data.get('club_location')

        for i, single_route in enumerate(data['routes']):

            param_filename = f'{club_slug}_route_{i}_police_api_param.txt'
            param_path = police_params_dir/param_filename

            metadata_path = metadata_dir/f'{club_slug}_route_{i}_metadata.json'

            try:
                param = build_police_api_param(
                route=single_route,
                club_name=club_name,
                area_name=area_name,
                version=version,
                corridor_width_m=corridor_width_m,
                )

                param_path.write_text(param)

                metadata = build_route_metadata(
                    area_name=area_name,
                    club_name=club_name,
                    club_location=club_location,
                    route=single_route,
                    route_index=i,
                    police_param_filename=param_filename,
                )

                with open(metadata_path, 'w') as f:
                    json.dump(metadata, f, indent=2)

                results[f"{club_name} | route {i}"] = {
                    "param_file": str(param_path),
                    "metadata_file": str(metadata_path),
                }

            except Exception as e:
                warnings.warn(
                    f"Failed to build Police API param for {club_name}: {e}"
                )

    return results

In [ ]:
batch_build_police_api_params(
    area_name='dalston',
    version='v1',
    corridor_width_m=40,
)

In [24]:
batch_build_police_api_params(
    area_name='Soho',
    version='v1',
    corridor_width_m=40,
)

{'Be At One - Regent Street | route 0': {'param_file': 'data/raw/v1/soho/police_params/be_at_one_-_regent_street_route_0_police_api_param.txt',
  'metadata_file': 'data/raw/v1/soho/metadata/be_at_one_-_regent_street_route_0_metadata.json'},
 'Be At One - Regent Street | route 1': {'param_file': 'data/raw/v1/soho/police_params/be_at_one_-_regent_street_route_1_police_api_param.txt',
  'metadata_file': 'data/raw/v1/soho/metadata/be_at_one_-_regent_street_route_1_metadata.json'},
 'Be At One - Regent Street | route 2': {'param_file': 'data/raw/v1/soho/police_params/be_at_one_-_regent_street_route_2_police_api_param.txt',
  'metadata_file': 'data/raw/v1/soho/metadata/be_at_one_-_regent_street_route_2_metadata.json'},
 'Be At One - Regent Street | route 3': {'param_file': 'data/raw/v1/soho/police_params/be_at_one_-_regent_street_route_3_police_api_param.txt',
  'metadata_file': 'data/raw/v1/soho/metadata/be_at_one_-_regent_street_route_3_metadata.json'},
 'Be At One - Regent Street | route 

## 6. Request Police API

In [25]:
def call_police_api(area_name, version, date):

    """
    Calls the UK Police API for each saved route polygon in an area,
    saving filtered crime data per route.
    """

    area_dir = Path(f'data/raw/{version}/{area_name.lower()}')
    params_dir = area_dir / 'police_params'

    if not params_dir.exists():
        raise FileNotFoundError(f"Area directory not found: {area_dir}")

    param_files = list(params_dir.glob("*_police_api_param.txt"))

    if not param_files:
        raise RuntimeError(f"No police API files found in {params_dir}")

    POLICE_URL = "https://data.police.uk/api/crimes-street/all-crime"

    allowed_categories = {
            "anti-social-behaviour",
            "violent-crime",
            "public-order"
        }

    output_dir = area_dir / 'crime_data'
    output_dir.mkdir(parents=True, exist_ok=True)

    for param_path in param_files:

        route_id = param_path.stem.replace('_police_api_param', '')   
        
        with open(param_path, 'r') as f:
            poly = f.read()

        params = {
            'date': date,
            'poly': poly,
        }
        
        response = requests.get(POLICE_URL, params=params)

        if response.status_code != 200:
            warnings.warn(
                f"Police API failed for {route_id}: {response.status_code}"
            )
            continue

        crimes = response.json()

        filtered_crimes = [
            c for c in crimes
            if c.get("category") in allowed_categories
        ]

        output_path = output_dir / f"{route_id}_crime_data_{date}.json"

        with open(output_path, 'w') as f:
            json.dump(filtered_crimes, f, indent=2)

In [34]:
AREA = 'Dalston'
VERSION = 'v1'
DATE = '2025-01'

call_police_api(area_name=AREA, version=VERSION, date=DATE)

In [26]:
AREA = 'Soho'
VERSION = 'v1'
DATE = '2025-01'

call_police_api(area_name=AREA, version=VERSION, date=DATE)

### Limitations and Reflections before NB02

_Data Quality of Police API_

The [Police API](https://data.police.uk/about/) lists some known issues regarding data quality. A particularly important issue for our purposes is that there are inconsistent geocoding policies in police forces. Estimates of geocoding accuracy in different forces range from 60% to 97%. We cannot filter for confident location data so we will acknowledge this limitation in our report.

_Police API Limitations_

The Police API does not allow for requests at a specific time; it only gives the month that the crime was commited. This will be acknowledged in our report.

_Google Route Matrix API Version_

The 'Walk' option of the [RouteTravelMode](https://developers.google.com/maps/documentation/routes/reference/rest/v2/RouteTravelMode?_gl=1*1v9v6vk*_up*MQ..*_ga*MTQ3MDM2MzQyLjE3NjcyMzk3MzQ.*_ga_NRWSTWS78N*czE3NjcyMzk3MzMkbzEkZzEkdDE3NjcyNDAyMTkkajYwJGwwJGgw*_ga_SM8HXJ53K2*czE3NjcyMzk3MzMkbzEkZzAkdDE3NjcyNDAyMTckajYwJGwwJGgw) is in beta. Google Maps state that it might sometimes be missing clear sidewalk and pedestrian paths. 

_Justifications_

The **downsampling** of the google routes polyline was necessary because the corridor polygon that we later compute creates more vertices (two opposite coordinates perpendicular to the polyline coordinates). Limiting the number of vertices is necessary for good API citizenship when requesting the Police API. 

The 'buffer polygon' created in part 4 maps the _actual route taken_: it is not simply an arbitrary box containing coordinates along the route.